## AlexNet

A CNN model based on AlexNet to classify 3 channel Oxford Flowers-102 dataset

https://www.robots.ox.ac.uk/~vgg/data/flowers/102/

In [2]:
import torch 
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [3]:
# Load Oxford Flowers dataset and create train/test split
from torchvision.datasets import Flowers102

# AlexNet expects 3-channel images around 224x224; normalize with ImageNet statistics
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

train_dataset = Flowers102(root="./data", split="train", download=True, transform=train_transform)
test_dataset = Flowers102(root="./data", split="test", download=True, transform=test_transform)

# Flowers102 has 102 classes
num_classes = 102

# Number of samples model processes at once during training/testing
batch_size = 32

# Create data loaders to handle batching and shuffling of data during training/testing
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))
print("Number of classes:", num_classes)

Train size: 1020
Test size: 6149
Number of classes: 102


In [12]:
model = nn.Sequential(
    # Block 1 with 96 filters, 11x11 kernel, stride 4
    nn.Conv2d(in_channels=3, out_channels=96, kernel_size=11, stride=4),
    nn.ReLU(inplace=True),
    nn.MaxPool2d(kernel_size=3, stride=2),
    nn.BatchNorm2d(96),

    # Block 2 with 256 filters, 5x5 kernel
    nn.Conv2d(96, 256, kernel_size=5),
    nn.ReLU(inplace=True),
    nn.MaxPool2d(kernel_size=3, stride=2),
    nn.BatchNorm2d(256),

    # Block 3 with 384 filters, 3x3 kernel
    nn.Conv2d(256, 256, kernel_size=3),
    nn.ReLU(inplace=True),
    nn.Conv2d(256, 384, kernel_size=3),
    nn.ReLU(inplace=True),
    nn.Conv2d(384, 384, kernel_size=3),
    nn.ReLU(inplace=True),
    nn.MaxPool2d(kernel_size=3, stride=2),
    nn.BatchNorm2d(384),

    # Classifier head
    nn.Flatten(),
    nn.Linear(384, 4096),
    nn.Tanh(),
    nn.Dropout(0.5),
    nn.Linear(4096, 4096),
    nn.Tanh(),
    nn.Dropout(0.5),
    nn.Linear(4096, num_classes),
    # CrossEntropyLoss applies softmax internally.
    # nn.Softmax(dim=1),
)

In [13]:
print(model)
total_params = sum(p.numel() for p in model.parameters())
print("Total model parameters:", total_params)

Sequential(
  (0): Conv2d(3, 96, kernel_size=(11, 11), stride=(4, 4))
  (1): ReLU(inplace=True)
  (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  (3): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (4): Conv2d(96, 256, kernel_size=(5, 5), stride=(1, 1))
  (5): ReLU(inplace=True)
  (6): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  (7): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (8): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1))
  (9): ReLU(inplace=True)
  (10): Conv2d(256, 384, kernel_size=(3, 3), stride=(1, 1))
  (11): ReLU(inplace=True)
  (12): Conv2d(384, 384, kernel_size=(3, 3), stride=(1, 1))
  (13): ReLU(inplace=True)
  (14): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  (15): BatchNorm2d(384, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (16): Flatten(start_dim=1, end_dim=-1)
  (17): L

In [14]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0002) # Adam optimizer 

In [15]:
# Model training loop with GPU support if available
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)
model = model.to(device)

epochs = 30
for epoch in range(epochs):
    model.train() # Set model to training mode (enables dropout, batch norm, etc. if present)
    running_loss = 0.0 
    correct = 0
    total = 0

    # Iterate over mini-batches from the training loader
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images) # Forward pass: compute predicted probabilities for the current batch
        loss = criterion(outputs, labels) # CrossEntropyLoss expects class indices, not one-hot labels
        loss.backward() # Backward pass: compute gradients of the loss with respect to model parameters
        optimizer.step() # Update model parameters based on computed gradients and learning rate

        # Track loss and accuracy for the current epoch
        running_loss += loss.item() * images.size(0)
        predicted = outputs.argmax(dim=1) # Get predicted class 
        correct += (predicted == labels).sum().item() # Count correct predictions in the current batch
        total += labels.size(0)

    train_loss = running_loss / total # Epoch average loss per sample
    train_acc = correct / total # Epoch training accuracy 

    # Evaluate on the test split without computing gradients
    model.eval() # Set model to evaluation mode (disables dropout, batch norm updates, etc.)
    test_correct = 0
    test_total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images) # Forward pass on test data to compute predicted probabilities
            predicted = outputs.argmax(dim=1) # Get predicted class
            test_correct += (predicted == labels).sum().item()
            test_total += labels.size(0)

    test_acc = test_correct / test_total


    print(f"Epoch {epoch + 1}/{epochs} - loss: {train_loss:.4f} - train acc: {train_acc:.4f} - test acc: {test_acc:.4f}")

Using device: mps
Epoch 1/30 - loss: 4.1858 - train acc: 0.0500 - test acc: 0.1160
Epoch 2/30 - loss: 3.4559 - train acc: 0.1304 - test acc: 0.1534
Epoch 3/30 - loss: 3.0948 - train acc: 0.2206 - test acc: 0.1656
Epoch 4/30 - loss: 2.8116 - train acc: 0.2824 - test acc: 0.2051
Epoch 5/30 - loss: 2.5943 - train acc: 0.3314 - test acc: 0.2233
Epoch 6/30 - loss: 2.2286 - train acc: 0.3833 - test acc: 0.2753
Epoch 7/30 - loss: 2.1142 - train acc: 0.4186 - test acc: 0.2441
Epoch 8/30 - loss: 1.7994 - train acc: 0.5255 - test acc: 0.2768
Epoch 9/30 - loss: 1.5578 - train acc: 0.5510 - test acc: 0.2976
Epoch 10/30 - loss: 1.2764 - train acc: 0.6578 - test acc: 0.3038
Epoch 11/30 - loss: 1.1517 - train acc: 0.6882 - test acc: 0.3101
Epoch 12/30 - loss: 0.8997 - train acc: 0.7539 - test acc: 0.3191
Epoch 13/30 - loss: 0.7323 - train acc: 0.8118 - test acc: 0.3534
Epoch 14/30 - loss: 0.6365 - train acc: 0.8333 - test acc: 0.3575
Epoch 15/30 - loss: 0.5218 - train acc: 0.8686 - test acc: 0.3513
E